# Module 5 / Track A1 — Statistical features on the WISDM dataset

Adapted from the course's earlier `data_exploration` notebook, which follows the Medium tutorial *Feature engineering on time-series data* (smartphone accelerometer HAR).

**Plan:** load & clean raw WISDM → EDA → split **by user** → window → statistical features → logistic-regression baseline → random forest → feature importance → retrain on top-10.

**Data:** download `WISDM_ar_v1.1_raw.txt` from https://www.cis.fordham.edu/wisdm/dataset.php into `./Data/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load and clean
The raw file is messy on purpose (real data usually is): bad lines, a trailing `;` on the z column, zero timestamps.

In [ ]:
columns = ['user', 'activity', 'timestamp', 'x-axis', 'y-axis', 'z-axis']
har_df = pd.read_csv('./Data/WISDM_ar_v1.1_raw.txt', on_bad_lines='skip',
                     header=None, names=columns)

har_df = har_df.dropna()
har_df['z-axis'] = har_df['z-axis'].astype(str).str.replace(';', '').astype(float)
df = har_df[har_df['timestamp'] != 0]
df = df.sort_values(by=['user', 'timestamp'], ignore_index=True)
df.describe()

## 2. Exploratory data analysis
Class balance, raw time series per activity, and value distributions.

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(x='activity', data=df, hue='activity')
plt.title('Number of samples by activity')
plt.show()

# 400 samples = 20 s at 20 Hz
for act in ['Walking', 'Jogging', 'Sitting']:
    d = df[(df['user'] == 36) & (df['activity'] == act)][:400]
    plt.figure(figsize=(14, 3))
    for ax in ['x-axis', 'y-axis', 'z-axis']:
        plt.plot(d['timestamp'].values, d[ax].values, label=ax)
    plt.title(act); plt.legend(); plt.show()

In [ ]:
for ax in ['x-axis', 'y-axis', 'z-axis']:
    g = sns.FacetGrid(df, hue='activity', height=3, aspect=3)
    g.map(sns.kdeplot, ax).add_legend()
    plt.show()

**Question:** based on the plots alone — which statistical quantities would separate *Jogging* from *Standing*? Which would separate *Upstairs* from *Downstairs* (harder!)? Write your guesses down; we check them against the feature importances at the end.

## 3. Train/test split — by user
Adjacent samples from one user are highly correlated. We want to generalise to *new users*, so we split along the user axis (train: users 1–27, test: 28–36). Same principle as Module 4's split-by-recording rule for the fan data.

In [ ]:
df_train = df[df['user'] <= 27]
df_test = df[df['user'] > 27]
print('Train:', df_train.shape, ' Test:', df_test.shape)

## 4. Windowing
100 samples @ 20 Hz = 5 s windows, 50 % overlap. Window label = most frequent activity inside the window.

In [ ]:
def make_windows(d, window_size=100, step_size=50):
    xs, ys, zs, labels = [], [], [], []
    for i in range(0, d.shape[0] - window_size, step_size):
        xs.append(d['x-axis'].values[i:i + window_size])
        ys.append(d['y-axis'].values[i:i + window_size])
        zs.append(d['z-axis'].values[i:i + window_size])
        labels.append(d['activity'][i:i + window_size].mode()[0])
    return xs, ys, zs, np.array(labels)

xw_tr, yw_tr, zw_tr, y_train = make_windows(df_train)
xw_te, yw_te, zw_te, y_test = make_windows(df_test)
print(len(xw_tr), 'train windows,', len(xw_te), 'test windows')

## 5. Statistical features
One function, one window, one axis → a dict of features. Keeping it in a function (instead of 59 copy-pasted lines) makes it reusable on the fan data in notebook 02 — and makes the C port in Track B a like-for-like exercise.

Note `std`: we use the **population** std (`numpy` default, divide by N). The C implementation uses the same convention — this is validation-relevant!

In [ ]:
def axis_features(w, prefix):
    w = np.asarray(w, dtype=float)
    mean = w.mean()
    f = {
        f'{prefix}_mean': mean,
        f'{prefix}_std': w.std(),                        # population std (/N)
        f'{prefix}_aad': np.mean(np.abs(w - mean)),      # average absolute deviation
        f'{prefix}_min': w.min(),
        f'{prefix}_max': w.max(),
        f'{prefix}_maxmin_diff': w.max() - w.min(),
        f'{prefix}_median': np.median(w),
        f'{prefix}_mad': np.median(np.abs(w - np.median(w))),
        f'{prefix}_IQR': np.percentile(w, 75) - np.percentile(w, 25),
        f'{prefix}_neg_count': int(np.sum(w < 0)),
        f'{prefix}_pos_count': int(np.sum(w > 0)),
        f'{prefix}_above_mean': int(np.sum(w > mean)),
        f'{prefix}_peak_count': len(find_peaks(w)[0]),
        f'{prefix}_skewness': stats.skew(w),
        f'{prefix}_kurtosis': stats.kurtosis(w),
        f'{prefix}_rms': np.sqrt(np.mean(w ** 2)),
        f'{prefix}_energy': np.sum(w ** 2) / len(w),
        f'{prefix}_zero_crossings': int(np.sum((w[:-1] - mean) * (w[1:] - mean) < 0)),
    }
    return f

def compute_features(xw, yw, zw):
    rows = []
    for wx, wy, wz in zip(xw, yw, zw):
        row = {}
        row.update(axis_features(wx, 'x'))
        row.update(axis_features(wy, 'y'))
        row.update(axis_features(wz, 'z'))
        res = np.sqrt(np.asarray(wx)**2 + np.asarray(wy)**2 + np.asarray(wz)**2)
        row['avg_result_accl'] = res.mean()
        row['sma'] = (np.abs(wx).sum() + np.abs(wy).sum() + np.abs(wz).sum()) / len(wx)
        rows.append(row)
    return pd.DataFrame(rows)

X_train = compute_features(xw_tr, yw_tr, zw_tr)
X_test = compute_features(xw_te, yw_te, zw_te)
print(X_train.shape, X_test.shape)
X_train.head()

## 6. Baseline: logistic regression
A linear model relies entirely on feature quality — if it scores well, the features carry the information. Expect ≈ 0.80.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

scaler = StandardScaler().fit(X_train)
lr = LogisticRegression(random_state=21, max_iter=1000)
lr.fit(scaler.transform(X_train), y_train)
y_pred = lr.predict(scaler.transform(X_test))
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

labels = sorted(np.unique(y_test))
sns.heatmap(confusion_matrix(y_test, y_pred, labels=labels),
            xticklabels=labels, yticklabels=labels, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Logistic regression'); plt.ylabel('True'); plt.xlabel('Predicted'); plt.show()

## 7. Random forest + feature importance
Trees need **no feature scaling** (thresholds are learned in raw units) — remember this for the C deployment in Module 6/7.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=15, random_state=21)
rf.fit(X_train, y_train)
print('RF accuracy:', accuracy_score(y_test, rf.predict(X_test)))

importances = pd.Series(rf.feature_importances_, index=X_train.columns) \
                .sort_values(ascending=False)
importances.head(15).plot.barh(figsize=(7, 5))
plt.gca().invert_yaxis(); plt.title('Top-15 feature importances'); plt.show()

In [ ]:
top10 = importances.index[:10]
print('Top-10 features:', list(top10))

rf_small = RandomForestClassifier(n_estimators=15, random_state=21)
rf_small.fit(X_train[top10], y_train)
y_pred_small = rf_small.predict(X_test[top10])
print('Top-10 RF accuracy:', accuracy_score(y_test, y_pred_small))

sns.heatmap(confusion_matrix(y_test, y_pred_small, labels=labels),
            xticklabels=labels, yticklabels=labels, annot=True, fmt='d', cmap='YlGnBu')
plt.title('RF, top-10 features only'); plt.ylabel('True'); plt.xlabel('Predicted'); plt.show()

## 8. Wrap-up questions
1. How much accuracy did dropping 46 of 56 features cost? Was it worth 6× less compute?
2. Which of the top-10 are cheap in C (single pass) and which need a sort or `find_peaks`?
3. Correlated features (`y_std`, `y_mad`, `y_IQR` all measure spread) split importance between them — how would you pick just one?
4. Did your guesses from the EDA question match the importance ranking?

→ Continue with **`02_fan_features.ipynb`**: same recipe on *your own* fan dataset, plus the FFT features this notebook deliberately skipped.